In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from matplotlib.path import Path
from tqdm import tqdm

def json_to_square_matrix(json_data, target_size=64, meter2pixel=100, border_pad=25, center=True):
    """
    Конвертирует JSON с вершинами в квадратную бинарную матрицу заданного размера.
    
    Параметры:
    - json_data: словарь с данными JSON
    - target_size: целевой размер стороны квадратной матрицы
    - meter2pixel: коэффициент преобразования метров в пиксели
    - border_pad: отступ от границ в пикселях
    - center: центрировать ли изображение в матрице
    """
    # Получаем вершины и масштабируем их
    verts = np.array(json_data["verts"]) * meter2pixel
    verts = verts.astype(int)
    
    # Используем bbox из JSON, если доступен
    if "bbox" in json_data:
        # Масштабируем минимальные и максимальные значения
        x_min = int(json_data["bbox"]["min"][0] * meter2pixel)
        y_min = int(json_data["bbox"]["min"][1] * meter2pixel)
        x_max = int(json_data["bbox"]["max"][0] * meter2pixel)
        y_max = int(json_data["bbox"]["max"][1] * meter2pixel)
    else:
        # Находим границы карты из вершин
        x_min, y_min = np.min(verts, axis=0)
        x_max, y_max = np.max(verts, axis=0)
    
    # Определяем размеры оригинальной матрицы
    orig_width = x_max - x_min + border_pad * 2
    orig_height = y_max - y_min + border_pad * 2
    
    # Смещаем вершины относительно минимальных координат и добавляем отступ
    verts[:, 0] = verts[:, 0] - x_min + border_pad
    verts[:, 1] = verts[:, 1] - y_min + border_pad
    
    # Определяем размер квадратной матрицы
    if max(orig_width, orig_height) < target_size:
        # Если исходный размер меньше целевого, используем целевой размер
        square_size = target_size
    else:
        # Используем ближайшую степень двойки, большую чем исходный размер
        square_size = target_size
    
    # Масштабируем вершины к целевому размеру напрямую
    scale = square_size / max(orig_width, orig_height)
    verts = (verts * scale).astype(int)
    
    # Центрируем изображение в квадратной матрице, если требуется
    if center:
        # Размеры масштабированного изображения
        scaled_width = int(orig_width * scale)
        scaled_height = int(orig_height * scale)
        
        # Вычисляем смещение для центрирования
        offset_x = (square_size - scaled_width) // 2
        offset_y = (square_size - scaled_height) // 2
        
        # Смещаем вершины
        verts[:, 0] += offset_x
        verts[:, 1] += offset_y
    
    # Создаем квадратную матрицу
    matrix = np.ones((square_size, square_size), dtype=np.uint8)
    
    # Рисуем линии стен прямо на матрице целевого размера
    for i in range(len(verts)):
        x1, y1 = verts[i]
        x2, y2 = verts[(i + 1) % len(verts)]
        
        # Проверяем корректность координат и рисуем линию
        if (0 <= x1 < square_size and 0 <= y1 < square_size and 
            0 <= x2 < square_size and 0 <= y2 < square_size):
            draw_line(matrix, x1, y1, x2, y2)
    
    # Используем Path для определения внутренних областей
    polygon_path = Path(verts)
    
    # Создаем сетку точек для проверки
    y_grid, x_grid = np.mgrid[:square_size, :square_size]
    points = np.column_stack((x_grid.ravel(), y_grid.ravel()))
    
    # Определяем, какие точки внутри многоугольника
    mask = polygon_path.contains_points(points)
    mask = mask.reshape(square_size, square_size)
    
    # Отмечаем внутренние области, сохраняя стены (0)
    inside_matrix = np.ones((square_size, square_size), dtype=np.uint8)
    inside_matrix[mask] = 1  # Внутренние точки
    inside_matrix[~mask] = 0  # Внешние точки
    
    # Совмещаем информацию о стенах и внутренних/внешних областях
    # Если точка была отмечена как стена (0), она остается стеной
    # Иначе используем информацию о внутренней/внешней области
    final_matrix = inside_matrix.copy()
    final_matrix[matrix == 0] = 0  # Сохраняем стены
    
    # Если требуется размер меньше созданной матрицы, выполняем уменьшение
    if square_size > target_size:
        final_matrix = downsample_matrix(final_matrix, target_size)
    
    return final_matrix

def draw_line(matrix, x0, y0, x1, y1):
    """
    Рисует линию между двумя точками на матрице (алгоритм Брезенхема).
    """
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx - dy
    
    while True:
        # Проверяем границы
        if 0 <= y0 < matrix.shape[0] and 0 <= x0 < matrix.shape[1]:
            matrix[y0, x0] = 0  # Стена = 0
        
        if x0 == x1 and y0 == y1:
            break
        
        e2 = 2 * err
        if e2 > -dy:
            err -= dy
            x0 += sx
        if e2 < dx:
            err += dx
            y0 += sy

def downsample_matrix(matrix, target_size):
    """
    Нерекурсивная функция для уменьшения размера матрицы до целевого размера.
    Сохраняет стены при уменьшении.
    """
    if matrix.shape[0] <= target_size:
        return matrix
    
    # Вычисляем размер блока для уменьшения
    block_size = matrix.shape[0] // target_size
    
    # Создаем матрицу целевого размера
    result = np.ones((target_size, target_size), dtype=np.uint8)
    
    # Для каждой ячейки результата проверяем соответствующий блок в исходной матрице
    for y in range(target_size):
        y_start = y * block_size
        y_end = min((y + 1) * block_size, matrix.shape[0])
        
        for x in range(target_size):
            x_start = x * block_size
            x_end = min((x + 1) * block_size, matrix.shape[1])
            
            # Вырезаем блок из исходной матрицы
            block = matrix[y_start:y_end, x_start:x_end]
            
            # Если в блоке есть стены, сохраняем стену
            if np.any(block == 0):
                result[y, x] = 0
    
    return result

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

def process_random_json_files(json_dir, output_npy_file, sample_size=5000, output_png_dir=None, matrix_size=64):
    """
    Случайно выбирает и обрабатывает JSON-файлы из директории и создает массив numpy и PNG-изображения.
    
    Параметры:
    - json_dir: директория с JSON-файлами
    - output_npy_file: имя файла для сохранения массива numpy
    - sample_size: количество файлов для случайной выборки
    - output_png_dir: директория для сохранения PNG-изображений (если None, PNG не сохраняются)
    - matrix_size: размер стороны квадратной матрицы
    """
    # Получаем список всех JSON-файлов в директории
    all_json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    total_files = len(all_json_files)
    
    print(f"Найдено {total_files} JSON-файлов в директории")
    
    # Выбираем случайную выборку файлов
    sample_size = min(sample_size, total_files)  # На случай, если sample_size больше, чем количество файлов
    sampled_files = random.sample(all_json_files, sample_size)
    
    print(f"Случайно выбрано {len(sampled_files)} файлов для обработки")
    
    # Извлекаем map_ids из имен файлов (удаляем расширение .json)
    map_ids = [os.path.splitext(file)[0] for file in sampled_files]
    
    # Создаем массив для хранения всех матриц
    all_matrices = np.zeros((len(map_ids), matrix_size, matrix_size), dtype=np.uint8)
    
    # Создаем директорию для PNG, если нужно
    if output_png_dir and not os.path.exists(output_png_dir):
        os.makedirs(output_png_dir)
    
    # Обрабатываем каждый JSON-файл
    processed_count = 0
    for i, map_id in enumerate(tqdm(map_ids, desc="Обработка файлов")):
        try:
            # Путь к JSON-файлу
            json_path = os.path.join(json_dir, f"{map_id}.json")
            
            # Проверяем, существует ли файл
            if not os.path.exists(json_path):
                print(f"Внимание: файл {json_path} не найден, пропускаем")
                continue
            
            # Загружаем JSON
            with open(json_path, 'r') as f:
                json_data = json.load(f)
            
            # Создаем матрицу
            matrix = json_to_square_matrix(
                json_data, 
                target_size=matrix_size,
                meter2pixel=100,
                border_pad=25,
                center=True
            )
            
            # Сохраняем матрицу в общий массив
            all_matrices[i] = matrix
            processed_count += 1
            
            # Если нужно, сохраняем PNG-изображение
            if output_png_dir:
                png_path = os.path.join(output_png_dir, f"{map_id}.png")
                
                # Создаем фигуру с высоким разрешением и точным размером
                plt.figure(figsize=(10, 10), dpi=300)
                
                # Настройка отображения матрицы
                plt.imshow(matrix, cmap='gray', interpolation='nearest')
                
                # Удаление осей и всех элементов интерфейса
                plt.axis('off')
                plt.xticks([])
                plt.yticks([])
                
                # Устранение всех полей вокруг изображения
                plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
                
                # Максимально плотная компоновка
                plt.tight_layout(pad=0)
                
                # Сохранение с высоким качеством
                plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=300,
                          format='png', transparent=False, facecolor='white', edgecolor='none')
                plt.close()
                
        except Exception as e:
            print(f"Ошибка при обработке {map_id}: {str(e)}")
            continue
    
    # Если были пропущены файлы, обрезаем массив до реального количества обработанных
    if processed_count < len(map_ids):
        all_matrices = all_matrices[:processed_count]
    
    # Сохраняем массив numpy
    np.save(output_npy_file, all_matrices)
    print(f"Массив сохранен в {output_npy_file}, форма: {all_matrices.shape}")
    
    return all_matrices

# Пример использования
process_random_json_files(
    json_dir="/home/silvarum/HouseExpo/HouseExpo/json",
    output_npy_file="/home/silvarum/TransPath_Adaptation/results/rooms_random",
    sample_size=5000,  # Выбрать 5000 случайных файлов
    output_png_dir="/home/silvarum/TransPath_Adaptation/results/rs_random",
    matrix_size=64
)

# from dataset_functions.dataset_generation.advanced_generator import find_bad_gen_indices

# def find_bad_gen_indices(T):
#     """
#     Проверяет, есть ли битые генерации во время генерации карт какого-то конкретного вида
#     """
#     bad_by_first_metric = reachability_variety(T)
#     bad_by_second_metric = reachability_hardness(T)
#     return np.array(list(set(bad_by_first_metric + bad_by_second_metric)))

# process_json_files(json_dir = "/home/silvarum/HouseExpo/HouseExpo/json", 
#                    id_file = "/home/silvarum/HouseExpo/HouseExpo/map_id_100.txt", 
#                    output_npy_file = "/home/silvarum/TransPath_Adaptation/results/rooms2", 
#                    output_png_dir="/home/silvarum/TransPath_Adaptation/results/rs2", 
#                    matrix_size=64)

# Пример использования

Найдено 100 ID для обработки


Обработка файлов: 100%|██████████| 100/100 [02:39<00:00,  1.59s/it]

Массив сохранен в /home/silvarum/TransPath_Adaptation/results/rooms2, форма: (100, 64, 64)


array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 1, 1, ..., 1, 0, 0],
        [0, 1, 1, ..., 1, 0, 0],
        ...,
        [0, 0, 0, ..., 1, 0, 0],
        [0, 0, 0, ..., 1, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 